In [ ]:
import os
from dotenv import load_dotenv
import time
import random
import pandas as pd
from pathlib import Path
from io import BytesIO
from concurrent.futures import ThreadPoolExecutor, as_completed
from PIL import Image, ImageFile
from google import genai
from google.genai import types

# .env 파일에 정의된 환경 변수를 불러옵니다.
load_dotenv()

# ==============================================================================
# [1. 설정 관리]
# ==============================================================================
class Config:
    # 환경 변수에서 API 키를 읽어옵니다. (보안 강화)
    Gemini_API_KEY = os.getenv("GEMINI_API_KEY")
    MODEL_ID = "gemini-3-pro-image-preview"

    # 경로 설정
    REF_A_DIR = Path("Ref A (Package)")
    REF_B_DIR = Path("Ref B (Logo)")
    REF_C_DIR = Path("Ref C (Font)")
    OUT_DIR   = Path("Output")

    # 생성 파라미터
    LIMIT = 12
    WORKERS = 4
    ASPECT_RATIO = "3:4"
    IMAGE_SIZE = "2K"
    TEMP = 0.2
    TOP_P = 0.8
    LOG_FILE = "generation_log.csv"

# API 키가 정상적으로 로드되었는지 확인 후 클라이언트 생성
if not Config.Gemini_API_KEY:
    raise ValueError("❌ .env 파일에서 GEMINI_API_KEY를 찾을 수 없습니다.")

ImageFile.LOAD_TRUNCATED_IMAGES = False
client = genai.Client(api_key=Config.Gemini_API_KEY)

# ==============================================================================
# [2. 유틸리티 함수]
# ==============================================================================
def get_images(path: Path):
    exts = {'.png', '.jpg', '.jpeg', '.webp'}
    if not path.exists(): return []
    return sorted([str(p) for p in path.rglob('*') if p.suffix.lower() in exts])

def safe_load(path: str, max_size=1024):
    try:
        with Image.open(path) as im:
            im.load()
            img = im.convert("RGB")
            if max(img.size) > max_size:
                img.thumbnail((max_size, max_size), Image.Resampling.LANCZOS)
            return img.copy()
    except: return None

def get_next_num(output_dir: Path, prefix="ref_"):
    nums = [int(p.stem.replace(prefix, "")) for p in output_dir.glob(f"{prefix}*.png") if p.stem.replace(prefix, "").isdigit()]
    return max(nums) + 1 if nums else 1

# ==============================================================================
# [3. 메인 프로세스]
# ==============================================================================
def run_task(b_path, l_path, f_path, out_dir, idx):
    fname = f"ref_{idx:04d}.png"
    save_p = out_dir / fname

    # CSV 저장용 데이터 구조
    res = {
        "status": "INIT",
        "generated_filename": fname,
        "bottle": Path(b_path).name,
        "logo": Path(l_path).name,
        "font": Path(f_path).name
    }

    try:
        if save_p.exists():
            return {**res, "status": "SKIP", "log_msg": f"이미 존재함: {fname}"}

        imgs = [safe_load(p) for p in [b_path, l_path, f_path]]
        if not all(imgs):
            return {**res, "status": "ERROR", "log_msg": f"❌ [{fname}] 입력 이미지 손상"}

        prompt = """
You are generating a single, high-end, photorealistic studio product image for a cosmetic brand.

CRITICAL REFERENCE ROLES (Zero Deviation Allowed):
- Reference A (STRUCTURE): Replicate the exact container shape, silhouette, cap design, material finish, and lighting reflections from Ref A. Do not alter the 3D geometry or proportions in any way.
- Reference B (LOGO): Extract the exact logo from Ref B without any loss of quality. Place it on the front upper-center area of the container. Apply a mathematically precise cylindrical wrap that perfectly follows the bottle's curvature without warping, stretching, or blurring.
- Reference C (TYPOGRAPHY): Transcribe all literal text and typographic layout exactly from Ref C. Render the text with absolute clarity and crispness. There must be zero pixelation, blurring, or illegible characters.

STRICT OUTPUT RULES:
- Hallucination Penalty: Do NOT render any extra text, words, slogans, graphics, or badges that are not explicitly present in Ref B and Ref C.
- Background: Keep the background completely pure white (#FFFFFF).
- Composition: Exactly one product perfectly centered in the frame.
""".strip()

        resp = client.models.generate_content(
            model=Config.MODEL_ID,
            contents=[prompt, *imgs],
            config=types.GenerateContentConfig(
                response_modalities=["TEXT", "IMAGE"],
                temperature=Config.TEMP,
                top_p=Config.TOP_P,
                image_config=types.ImageConfig(
                    aspect_ratio=Config.ASPECT_RATIO,
                    image_size=Config.IMAGE_SIZE
                )
            )
        )

        for cand in resp.candidates:
            if cand.content and cand.content.parts:
                for part in cand.content.parts:
                    if part.inline_data:
                        Image.open(BytesIO(part.inline_data.data)).convert("RGB").save(save_p)
                        return {**res, "status": "SUCCESS", "log_msg": f"✅ [{fname}] 생성 완료"}

        return {**res, "status": "ERROR", "log_msg": f"❌ [{fname}] 생성 실패 (응답에 이미지 없음)"}

    except Exception as e:
        # [수정] CSV 상태는 'ERROR'로 고정, 터미널(log_msg)에만 상세 정보 출력
        return {**res, "status": "ERROR", "log_msg": f"❌ [{fname}] 오류 발생: {str(e)}"}

def main():
    Config.OUT_DIR.mkdir(exist_ok=True)
    start_idx = get_next_num(Config.OUT_DIR)

    # 리소스 풀 로딩
    pools = {name: get_images(Config.REF_C_DIR / name) for name in ["front_bw", "front_color", "back_bw", "back_color"]}
    bottles, logos = get_images(Config.REF_A_DIR), get_images(Config.REF_B_DIR)

    if not bottles or not logos:
        print("❌ Ref A 또는 Ref B 이미지가 없습니다. 경로를 확인해주세요.")
        return

    tasks, used_f = [], set()
    print(f"🎲 작업 매칭 시작 (목표: {Config.LIMIT}개)...")

    while len(tasks) < Config.LIMIT:
        is_front = len(tasks) % 2 == 0
        bottle = random.choice(bottles)
        is_dark = "DARK" in Path(bottle).parts

        p_keys = (["front_color"] if is_dark else ["front_bw", "front_color"]) if is_front else \
                 (["back_color"] if is_dark else ["back_bw", "back_color"])

        combined_pool = [f for k in p_keys if k in pools for f in pools[k]]
        if not combined_pool: continue

        font = random.choice(combined_pool)
        if font in used_f: continue
        used_f.add(font)

        tasks.append((bottle, random.choice(logos), font, Config.OUT_DIR, start_idx + len(tasks)))

    print(f"🚀 이미지 생성 시작 (병렬 처리: {Config.WORKERS})...")
    results_for_csv = []

    with ThreadPoolExecutor(max_workers=Config.WORKERS) as exe:
        futures = [exe.submit(run_task, *t) for t in tasks]
        for f in as_completed(futures):
            res = f.result()
            # 터미널에 상세 로그 출력 후, CSV 데이터에서 제거
            print(res.pop("log_msg", "알 수 없는 작업 완료"))
            results_for_csv.append(res)

    # CSV 저장 (status 컬럼에는 'ERROR'만 기록됨)
    df = pd.DataFrame(results_for_csv)
    cols = ["status", "generated_filename", "bottle", "logo", "font"]
    df = df.reindex(columns=cols)

    csv_p = Config.OUT_DIR / Config.LOG_FILE
    # 새로고침을 위해 기존 파일을 삭제하고 싶으시면 아래 주석을 해제하세요.
    # if csv_p.exists(): csv_p.unlink()

    df.to_csv(csv_p, mode='a' if csv_p.exists() else 'w', header=not csv_p.exists(), index=False, encoding='utf-8-sig')
    print(f"\n💾 로그 저장 완료: {csv_p}")

if __name__ == "__main__":
    main()